# Notebook 6: Bayesian A/B Testing

## Overview
This notebook introduces **Bayesian thinking** for A/B testing—a fundamentally different philosophical approach from the frequentist methods in Notebooks 2-5.

### Learning Objectives
- Understand Bayesian inference: prior + data → posterior
- Learn the Beta-Binomial model for binary outcomes
- Calculate probability of being best variant
- Decide using expected loss framework
- Compare Bayesian vs Frequentist approaches

### Why Bayesian?
1. **Intuitive interpretation**: "What's the probability this variant is better?" (exactly what practitioners want)
2. **Incorporates prior knowledge**: Can use historical data or domain expertise
3. **Sequential-friendly**: Easy to update beliefs as data arrives
4. **Decision-focused**: Built for making business decisions, not just statistical testing

### A Simple Analogy
- **Frequentist**: "If the coin were truly fair, how likely is this many heads?" (probability of data given hypothesis)
- **Bayesian**: "Given I saw this many heads, what's the probability the coin is fair?" (probability of hypothesis given data)

The Bayesian answer is usually what people actually want.

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams['figure.figsize'] = (12, 6)

# Create output directory
os.makedirs('../data/outputs/nb06', exist_ok=True)


In [ ]:
# Load cleaned data
data_path = Path("../data/outputs/nb01/nb01_hillstrom_clean.csv")
df = pd.read_csv(data_path)

print(f"Data shape: {df.shape}")
print(f"\nGroup stats:\n{df.groupby('segment')['conversion'].agg(['sum', 'count', 'mean'])}")

## Concept: Bayesian vs Frequentist Thinking

### Side-by-Side Comparison

| Aspect | Frequentist | Bayesian |
|--------|-------------|----------|
| **Probability** | Long-run frequency | Degree of belief |
| **Parameter** | Fixed unknown constant | Random variable with distribution |
| **Prior** | Irrelevant (or wrong to use) | Core of inference |
| **Inference** | P(data \| H₀) — probability of data under null | P(H \| data) — probability of hypothesis given data |
| **Conclusion** | "Reject H₀" or "Fail to reject" | "Parameter is likely in range [a, b]" |
| **Question** | "How often would this occur if H₀ true?" | "What do I believe about the parameter?" |
| **Early stopping** | Inflates Type I error | No problem! Just update beliefs |
| **CI interpretation** | Long-run coverage property | Direct probability interval |

### Example: Coin Flip
You flip a coin 10 times, get 8 heads. Is it fair?

**Frequentist**: P(8+ heads \| fair coin) ≈ 0.055. At α=0.05, just barely fail to reject fairness.

**Bayesian**: P(coin is fair \| 8 heads) is low. Prior belief matters:
- If you believed it was fair: "I'm now skeptical"
- If you knew it might be biased: "This confirms it's biased"

### Why Bayesian is Natural for A/B Testing
In business, we don't care about the probability of data under H₀. We care about:
- "What's P(variant A is better than variant B)?"
- "What's the expected loss if I pick variant A?"
- "Can I stop early and declare a winner?"

All of these are natural Bayesian questions.

## The Beta Distribution as a Prior

### What is Beta?
The **Beta distribution** is a flexible probability distribution on [0, 1], perfect for modeling uncertainty about a probability.

Parameters:
- **α** (alpha): shape parameter, roughly "successes + 1"
- **β** (beta): shape parameter, roughly "failures + 1"
- **Mean** = α / (α + β)
- **Variance** decreases with larger α + β (more concentrated)

### Intuitive Interpretation
Think of Beta(α, β) as encoding: "I've seen α successes and β failures."

- **Beta(1, 1)**: Uniform — "completely uninformed"
- **Beta(10, 10)**: Concentrated at 0.5 — "I'm pretty sure it's around 50%"
- **Beta(100, 100)**: Tightly concentrated at 0.5 — "I'm very confident it's ~50%"
- **Beta(90, 10)**: Peaked at 0.9 — "I believe it's around 90%, pretty confident"

### Why Conjugate?
Beta is the **conjugate prior** for binomial data. This means:
- **Prior**: Beta(α, β)
- **Data**: X successes out of n trials
- **Posterior**: Beta(α + X, β + n - X)

**Conjugacy is powerful**: the posterior has the same form as the prior! No complex math needed.

In [ ]:
# Visualize different Beta priors
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

x = np.linspace(0, 1, 1000)

priors = [
    (1, 1, "Beta(1,1)\nUninformed"),
    (5, 5, "Beta(5,5)\nModerate (50%)"),
    (10, 2, "Beta(10,2)\nInformative (83%)")
]

for ax, (alpha, beta, title) in zip(axes, priors):
    prior = stats.beta(alpha, beta)
    y = prior.pdf(x)
    
    ax.fill_between(x, y, alpha=0.6, color='steelblue')
    ax.plot(x, y, color='steelblue', linewidth=2)
    
    ax.set_xlabel('Probability', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)
    
    mean = alpha / (alpha + beta)
    ax.axvline(mean, color='red', linestyle='--', linewidth=1.5, label=f'Mean={mean:.2f}')
    ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('../data/outputs/nb06/nb06_beta_priors.png', dpi=300, bbox_inches='tight')
plt.show()

print("Beta prior distributions visualized.")

## Beta-Binomial Model for Conversion

### The Model
For each variant, we model:
1. **Prior**: Beta(α₀, β₀) — what we believe before seeing data
2. **Likelihood**: Binomial(p | X successes, n trials)
3. **Posterior**: Beta(α₀ + X, β₀ + n - X) — updated belief

### Interpretation
After observing data:
- **Posterior α** = prior α + observed successes
- **Posterior β** = prior β + observed failures

The posterior is centered at:
(α₀ + X) / (α₀ + β₀ + n)

With weak prior (α₀ = β₀ = 1), this is approximately the sample conversion rate.
With strong prior (α₀ = β₀ = 100), the posterior is pulled toward 0.5 (the prior belief).

### Advantages
- **No distributional assumptions**: Works exactly for binary outcomes
- **Closed-form posterior**: Easy calculation
- **Natural uncertainty**: The posterior spread captures sampling uncertainty
- **Easy updates**: New data → new posterior (which becomes next prior)

In [ ]:
# Calculate posteriors for each group using Beta-Binomial model
# Prior: weakly informative Beta(1, 1)

groups = df['segment'].unique()
posterior_params = {}

print("=== Beta-Binomial Posterior Analysis (Conversion) ===\n")

for group in groups:
    group_data = df[df['segment'] == group]
    conversions = group_data['conversion'].sum()
    failures = len(group_data) - conversions
    
    # Prior
    alpha_prior = 1
    beta_prior = 1
    
    # Posterior
    alpha_post = alpha_prior + conversions
    beta_post = beta_prior + failures
    
    posterior_params[group] = (alpha_post, beta_post)
    
    # Posterior mean and credible interval
    post_mean = alpha_post / (alpha_post + beta_post)
    post_var = (alpha_post * beta_post) / ((alpha_post + beta_post)**2 * (alpha_post + beta_post + 1))
    post_std = np.sqrt(post_var)
    
    # 95% HDI (High Density Interval) using quantiles
    post_dist = stats.beta(alpha_post, beta_post)
    hdi_lower = post_dist.ppf(0.025)
    hdi_upper = post_dist.ppf(0.975)
    
    print(f"{group}:")
    print(f"  Data: {conversions} conversions / {len(group_data)} customers")
    print(f"  Posterior: Beta({alpha_post}, {beta_post})")
    print(f"  Posterior mean: {post_mean:.4f}")
    print(f"  Posterior std: {post_std:.4f}")
    print(f"  95% HDI: [{hdi_lower:.4f}, {hdi_upper:.4f}]")
    print()

# Store for later use
posterior_df = pd.DataFrame([
    {
        'Segment': group,
        'Alpha': posterior_params[group][0],
        'Beta': posterior_params[group][1],
        'Mean': posterior_params[group][0] / (posterior_params[group][0] + posterior_params[group][1]),
        'HDI_Lower': stats.beta(posterior_params[group][0], posterior_params[group][1]).ppf(0.025),
        'HDI_Upper': stats.beta(posterior_params[group][0], posterior_params[group][1]).ppf(0.975)
    }
    for group in groups
])

print(posterior_df.to_string(index=False))

In [ ]:
# Visualize prior vs posterior for each group
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

x = np.linspace(0, 0.15, 1000)

for ax, group in zip(axes, groups):
    # Prior (Beta(1,1))
    prior = stats.beta(1, 1)
    
    # Posterior
    alpha_post, beta_post = posterior_params[group]
    posterior = stats.beta(alpha_post, beta_post)
    
    # Plot
    ax.fill_between(x, prior.pdf(x), alpha=0.4, color='lightblue', label='Prior (Beta 1,1)')
    ax.plot(x, prior.pdf(x), color='blue', linewidth=2)
    
    ax.fill_between(x, posterior.pdf(x), alpha=0.6, color='coral', label='Posterior')
    ax.plot(x, posterior.pdf(x), color='darkred', linewidth=2.5)
    
    # Mark posterior mean
    post_mean = alpha_post / (alpha_post + beta_post)
    ax.axvline(post_mean, color='darkred', linestyle='--', linewidth=1.5, alpha=0.7)
    
    ax.set_xlabel('Conversion Rate', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f"{group}", fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/outputs/nb06/nb06_posterior_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nPosterior distributions visualized.")

## Probability of Being Best

### The Key Insight
Instead of hypothesis testing (yes/no), Bayesian A/B testing gives us probabilities:
- P(Mens Email is best)
- P(Womens Email is best)
- P(No Email is best)

These sum to 100% and directly answer business questions.

### Method
1. Draw samples from each variant's posterior
2. For each sample set, determine which is highest
3. Count the frequency

This is **Monte Carlo estimation** and is fast even for many variants.

In [ ]:
# Monte Carlo: Draw samples from posteriors and calculate probabilities
np.random.seed(42)

n_samples = 100000

# Draw samples from each posterior
samples = {}
for group in groups:
    alpha, beta = posterior_params[group]
    samples[group] = np.random.beta(alpha, beta, n_samples)

# Calculate probabilities
prob_best = {}

for group in groups:
    # Count how many times this group has the highest conversion rate
    is_best = samples[group] > samples[[g for g in groups if g != group][0]]
    
    for other_group in groups:
        if other_group != group:
            is_best = is_best & (samples[group] > samples[other_group])
    
    prob_best[group] = np.mean(is_best)

print("\n=== Probability of Being Best (Monte Carlo) ===\n")
for group in groups:
    print(f"{group}: {prob_best[group]:.4f} ({prob_best[group]*100:.2f}%)")

# Pairwise comparisons
print("\n=== Pairwise Comparisons ===\n")

for i, group1 in enumerate(groups):
    for group2 in groups[i+1:]:
        prob_g1_better = np.mean(samples[group1] > samples[group2])
        prob_g2_better = np.mean(samples[group2] > samples[group1])
        
        print(f"P({group1} > {group2}): {prob_g1_better:.4f}")
        print(f"P({group2} > {group1}): {prob_g2_better:.4f}")
        print()

In [ ]:
# Visualize overlapping posteriors
fig, ax = plt.subplots(figsize=(12, 6))

x = np.linspace(0, 0.15, 1000)
colors = sns.color_palette("husl", len(groups))

for group, color in zip(groups, colors):
    alpha, beta = posterior_params[group]
    posterior = stats.beta(alpha, beta)
    
    ax.fill_between(x, posterior.pdf(x), alpha=0.4, color=color)
    ax.plot(x, posterior.pdf(x), color=color, linewidth=2.5, label=group)
    
    # Mark mean
    post_mean = alpha / (alpha + beta)
    ax.axvline(post_mean, color=color, linestyle='--', linewidth=1.5, alpha=0.7)

ax.set_xlabel('Conversion Rate', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Posterior Distributions with Probability of Being Best\n(Overlapping region shows uncertainty)', 
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# Add text box with probabilities
textstr = 'Probability of Being Best:
'
for group in groups:
    textstr += f"{group}: {prob_best[group]*100:.1f}%
"

ax.text(0.98, 0.97, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig('../data/outputs/nb06/nb06_bayesian_prob_best.png', dpi=300, bbox_inches='tight')
plt.show()

## Expected Loss: A Decision Framework

### The Idea
For each variant, we can compute the **expected loss** of choosing it:

**E[Loss(choosing A)] = ∫ Loss(A vs true param) × P(true param) d(param)**

A simple loss function: Loss = max(0, best_variant - chosen_variant)

If the best variant has conversion rate 0.05 and we choose a variant with 0.04, the loss is 0.01.

### Why This Matters
- Accounts for probability of being best AND the magnitude of potential loss
- Handles multiple variants naturally
- Focuses on what matters: minimizing regret, not just hypothesis testing

In [ ]:
# Calculate expected loss for each variant
def expected_loss(chosen_variant_samples, other_samples_list):
    """
    Calculate expected loss if we choose a given variant.
    Loss = max(0, max(others) - chosen)
    """
    max_other = np.max(np.array(other_samples_list), axis=0)
    loss = np.maximum(0, max_other - chosen_variant_samples)
    return np.mean(loss)

print("=== Expected Loss Analysis ===\n")

expected_losses = {}

for group in groups:
    other_samples = [samples[g] for g in groups if g != group]
    exp_loss = expected_loss(samples[group], other_samples)
    expected_losses[group] = exp_loss
    
    print(f"{group}: {exp_loss:.6f}")

# Identify best variant by expected loss
best_variant = min(expected_losses, key=expected_losses.get)
print(f"\nBest choice by expected loss: {best_variant}")

# Visualize expected loss
fig, ax = plt.subplots(figsize=(10, 6))

exp_loss_values = [expected_losses[g] for g in groups]
colors_loss = ['red' if g == best_variant else 'steelblue' for g in groups]

bars = ax.bar(groups, exp_loss_values, color=colors_loss, alpha=0.7, edgecolor='black', linewidth=1.5)

ax.set_ylabel('Expected Loss', fontsize=12)
ax.set_title('Expected Loss: Which Variant Should You Choose?\n(Lower is better)', 
             fontsize=13, fontweight='bold')
ax.grid(alpha=0.3, axis='y')

# Add value labels
for bar, loss in zip(bars, exp_loss_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.00002, 
            f'{loss:.6f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/outputs/nb06/nb06_expected_loss.png', dpi=300, bbox_inches='tight')
plt.show()

## Credible Intervals vs Confidence Intervals

### Key Difference
- **Confidence Interval (Frequentist)**: "If we repeated this experiment many times, 95% of our intervals would contain the true parameter"
  - The parameter is fixed, the interval is random
  - Probability is about the *procedure*, not this specific interval

- **Credible Interval (Bayesian)**: "Given the data, there's a 95% probability the parameter is in this interval"
  - The parameter is random (has a distribution), the interval is fixed
  - Probability is directly about the parameter value

### Highest Density Interval (HDI)
The Bayesian credible interval that includes the highest posterior density. For our Beta posteriors, we can compute this directly.

In [ ]:
# Compare CIs and credible intervals
ci_comparison = []

for group in groups:
    alpha_post, beta_post = posterior_params[group]
    posterior = stats.beta(alpha_post, beta_post)
    
    # Bayesian HDI (95%)
    hdi_lower = posterior.ppf(0.025)
    hdi_upper = posterior.ppf(0.975)
    
    # For comparison, analytical Wald CI (from earlier notebooks)
    group_data = df[df['segment'] == group]
    conversions = group_data['conversion'].sum()
    n = len(group_data)
    p = conversions / n
    se = np.sqrt(p * (1 - p) / n)
    z = stats.norm.ppf(0.975)
    
    wald_lower = p - z * se
    wald_upper = p + z * se
    
    ci_comparison.append({
        'Segment': group,
        'Bayesian HDI Lower': f"{hdi_lower:.4f}",
        'Bayesian HDI Upper': f"{hdi_upper:.4f}",
        'Wald CI Lower': f"{wald_lower:.4f}",
        'Wald CI Upper': f"{wald_upper:.4f}",
        'HDI Width': f"{hdi_upper - hdi_lower:.4f}",
        'Wald Width': f"{wald_upper - wald_lower:.4f}"
    })

ci_df = pd.DataFrame(ci_comparison)
print("\n=== Bayesian Credible Interval vs Frequentist CI ===\n")
print(ci_df.to_string(index=False))

# Save comparison
ci_df.to_csv('../data/outputs/nb06/nb06_ci_bayesian_comparison.csv', index=False)

## Sensitivity Analysis: Prior Choice

One criticism of Bayesian methods: "Results depend on the prior!"

This is partly true, but with large samples, the prior matters less. Let's test this.

In [ ]:
# Test multiple priors
priors_to_test = [
    (1, 1, "Uninformed Beta(1,1)"),
    (5, 5, "Weak Beta(5,5)"),
    (10, 10, "Moderate Beta(10,10)"),
    (20, 20, "Strong Beta(20,20)")
]

sensitivity_results = []

for alpha_prior, beta_prior, prior_name in priors_to_test:
    print(f"\nPrior: {prior_name}")
    print("-" * 50)
    
    prior_probs_best = {}
    
    for group in groups:
        group_data = df[df['segment'] == group]
        conversions = group_data['conversion'].sum()
        failures = len(group_data) - conversions
        
        # Posterior with this prior
        alpha_post = alpha_prior + conversions
        beta_post = beta_prior + failures
        
        # Sample
        samp = np.random.beta(alpha_post, beta_post, 50000)
        prior_probs_best[group] = samp
    
    # Calculate prob best
    prob_best_prior = {}
    for group in groups:
        is_best = prior_probs_best[group] > prior_probs_best[[g for g in groups if g != group][0]]
        for other in groups:
            if other != group:
                is_best = is_best & (prior_probs_best[group] > prior_probs_best[other])
        prob_best_prior[group] = np.mean(is_best)
    
    for group in groups:
        sensitivity_results.append({
            'Prior': prior_name,
            'Segment': group,
            'P(Best)': f"{prob_best_prior[group]:.4f}"
        })
    
    print(f"P(Men's Best): {prob_best_prior[groups[0]]:.4f}")
    if len(groups) > 1:
        print(f"P(Women's Best): {prob_best_prior[groups[1]]:.4f}")
    if len(groups) > 2:
        print(f"P(Control Best): {prob_best_prior[groups[2]]:.4f}")

sensitivity_df = pd.DataFrame(sensitivity_results)

print("\n\n=== SENSITIVITY ANALYSIS SUMMARY ===\n")
# Pivot for readability
pivot_sensitivity = sensitivity_df.pivot(index='Segment', columns='Prior', values='P(Best)')
print(pivot_sensitivity)

print("\n\nConclusion: With large samples (64K customers), prior choice has minimal impact.")
print("P(Best) is similar across all reasonable priors.")

## Extension: Bayesian Analysis for Spend (Normal-Normal Model)

For continuous outcomes like spend, we use a **Normal-Normal model**:
- **Prior**: N(μ₀, σ₀²)
- **Likelihood**: N(ȳ, σ²/n)
- **Posterior**: N(μₙ, σₙ²) with:
  - μₙ = (σ₀⁻² μ₀ + σ⁻² ȳ n) / (σ₀⁻² + σ⁻² n)
  - σₙ² = 1 / (σ₀⁻² + σ⁻² n)

The posterior mean is a weighted average of prior mean and observed sample mean.

In [ ]:
# Bayesian analysis for spend
# Prior: N(mean_spend, sd_spend) based on control group
control_data = df[df['segment'] == "No E-Mail"]
prior_mean = control_data['spend'].mean()
prior_sd = control_data['spend'].std()

print(f"\n=== Bayesian Analysis for Spend ===\n")
print(f"Prior (from control): N({prior_mean:.2f}, {prior_sd:.2f}²)")
print()

spend_results = []

for group in groups:
    group_data = df[df['segment'] == group]
    y_bar = group_data['spend'].mean()
    y_sd = group_data['spend'].std()
    n = len(group_data)
    
    # Posterior parameters
    prior_prec = 1 / (prior_sd ** 2)
    likelihood_prec = n / (y_sd ** 2)
    
    posterior_prec = prior_prec + likelihood_prec
    posterior_mean = (prior_prec * prior_mean + likelihood_prec * y_bar) / posterior_prec
    posterior_sd = 1 / np.sqrt(posterior_prec)
    
    # 95% credible interval
    z = stats.norm.ppf(0.975)
    ci_lower = posterior_mean - z * posterior_sd
    ci_upper = posterior_mean + z * posterior_sd
    
    print(f"{group}:")
    print(f"  Sample mean: ${y_bar:.2f}")
    print(f"  Posterior: N({posterior_mean:.2f}, {posterior_sd:.2f}²)")
    print(f"  95% Credible Interval: [${ci_lower:.2f}, ${ci_upper:.2f}]")
    print()
    
    spend_results.append({
        'Segment': group,
        'Sample Mean': y_bar,
        'Posterior Mean': posterior_mean,
        'Posterior SD': posterior_sd,
        'CI Lower': ci_lower,
        'CI Upper': ci_upper
    })

spend_df = pd.DataFrame(spend_results)

In [ ]:
# Create comprehensive Bayesian results summary
print("\n\n=== BAYESIAN A/B TEST SUMMARY ===\n")

summary_bayesian = pd.DataFrame([
    {
        'Segment': group,
        'N': len(df[df['segment'] == group]),
        'Conv Rate': f"{df[df['segment'] == group]['conversion'].mean():.4f}",
        'P(Best)': f"{prob_best[group]:.4f}",
        'Expected Loss': f"{expected_losses[group]:.6f}",
        'Recommended': 'YES' if group == best_variant else ''
    }
    for group in groups
])

print(summary_bayesian.to_string(index=False))

# Save results
summary_bayesian.to_csv('../data/outputs/nb06/nb06_bayesian_results.csv', index=False)

print("\n\n=== KEY INSIGHTS ===\n")
print(f"1. {best_variant} has lowest expected loss → Recommended choice")
print(f"\n2. Probability interpretation:")
for group in groups:
    print(f"   {group}: {prob_best[group]*100:.1f}% chance it's the best")

print(f"\n3. Full uncertainty captured by posterior distributions")
print(f"\n4. Sequential analysis-friendly: update as data arrives")
print(f"\n5. Results published to: ../data/outputs/nb06/nb06_bayesian_results.csv")

## Summary: Bayesian vs Frequentist for A/B Testing

### Frequentist Approach (Notebooks 2-5)
**Strengths**:
- Objective (no prior choice)
- Well-established, understood by regulators
- Controls Type I error rigorously

**Weaknesses**:
- Answers the "wrong" question (probability of data vs probability of hypothesis)
- Hypothesis testing is binary (reject/fail to reject) — not useful for business
- Early stopping inflates error rates
- Confidence intervals don't directly mean what people think

### Bayesian Approach (This Notebook)
**Strengths**:
- Direct answer to business questions: "Which variant is best?"
- Intuitive interpretation of intervals
- Sequential analysis is natural and unbiased
- Incorporates prior knowledge
- Provides probabilities of each outcome

**Weaknesses**:
- Requires choosing a prior (though impact diminishes with data)
- Computationally more involved (though usually fast)
- Less familiar to some practitioners

### When to Use Each
- **Bayesian**: Business decisions, online experiments, when you want interpretable probabilities
- **Frequentist**: Regulatory requirements, pre-registered designs, when prior information is unavailable
- **Best Practice**: Use both! Agreement across methods increases confidence.